# 💳 Credit Scoring Model
## CodeAlpha Machine Learning Internship — Task 1
**Author:** Rose Sharma  
**Objective:** Predict an individual's creditworthiness using past financial data  
**Algorithms:** Logistic Regression, Decision Tree, Random Forest  

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)
import pickle

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries imported!')

## 📂 Step 2 — Generate / Load Dataset

In [ ]:
# Generate realistic credit scoring dataset
np.random.seed(42)
n = 5000

age             = np.random.randint(21, 70, n)
income          = np.random.randint(15000, 200000, n)
loan_amount     = np.random.randint(1000, 100000, n)
loan_term       = np.random.choice([12, 24, 36, 48, 60], n)
credit_history  = np.random.randint(0, 10, n)  # years
num_credit_lines= np.random.randint(1, 15, n)
debt_to_income  = np.round(np.random.uniform(0.05, 0.75, n), 2)
missed_payments = np.random.randint(0, 10, n)
employment_yrs  = np.random.randint(0, 30, n)
existing_loans  = np.random.randint(0, 5, n)

# Target: creditworthy (1) or not (0)
score = (
    (income / 10000) * 0.3
    + credit_history * 0.25
    - missed_payments * 0.35
    - debt_to_income * 2
    + employment_yrs * 0.1
    - existing_loans * 0.15
    + np.random.normal(0, 0.5, n)
)
creditworthy = (score > score.mean()).astype(int)

df = pd.DataFrame({
    'age':              age,
    'income':           income,
    'loan_amount':      loan_amount,
    'loan_term':        loan_term,
    'credit_history':   credit_history,
    'num_credit_lines': num_credit_lines,
    'debt_to_income':   debt_to_income,
    'missed_payments':  missed_payments,
    'employment_years': employment_yrs,
    'existing_loans':   existing_loans,
    'creditworthy':     creditworthy
})

df.to_csv('credit_data.csv', index=False)
print(f'✅ Dataset created: {df.shape}')
print(f'   Creditworthy    : {creditworthy.sum():,} ({creditworthy.mean()*100:.1f}%)')
print(f'   Not Creditworthy: {(1-creditworthy).sum():,} ({(1-creditworthy).mean()*100:.1f}%)')
df.head()

## 🔍 Step 3 — EDA

In [ ]:
print('📊 Dataset Info:')
print(df.describe().round(2))
print(f'\nMissing values: {df.isnull().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Feature Distributions by Creditworthiness', fontsize=14, fontweight='bold')
axes = axes.flatten()
features = [c for c in df.columns if c != 'creditworthy']
colors = ['#e74c3c', '#27ae60']

for i, feat in enumerate(features):
    for label, color in zip([0, 1], colors):
        axes[i].hist(df[df['creditworthy']==label][feat],
                     bins=20, alpha=0.6, color=color,
                     label=['Not Creditworthy','Creditworthy'][label])
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## ✂️ Step 4 — Feature Engineering & Split

In [ ]:
# Feature engineering
df['income_to_loan']  = df['income'] / df['loan_amount']
df['monthly_payment'] = df['loan_amount'] / df['loan_term']
df['payment_burden']  = df['monthly_payment'] / (df['income'] / 12)

X = df.drop('creditworthy', axis=1)
y = df['creditworthy']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')

## 🤖 Step 5 — Train & Compare 3 Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_pred = m.predict(X_test_s)
    y_prob = m.predict_proba(X_test_s)[:, 1]
    results[name] = {
        'model':     m,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }
    print(f'\n✅ {name}')
    print(f'   Accuracy : {results[name]["accuracy"]:.4f}')
    print(f'   Precision: {results[name]["precision"]:.4f}')
    print(f'   Recall   : {results[name]["recall"]:.4f}')
    print(f'   F1-Score : {results[name]["f1"]:.4f}')
    print(f'   ROC-AUC  : {results[name]["roc_auc"]:.4f}')

In [ ]:
# ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Model Comparison', fontsize=14, fontweight='bold')
colors_roc = ['#1a3c6e', '#e74c3c', '#27ae60']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC={res["roc_auc"]:.3f})')

axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models')
axes[0].legend(loc='lower right')

# Metrics bar chart
metrics_names = ['accuracy','precision','recall','f1','roc_auc']
x = np.arange(len(metrics_names))
w = 0.25
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics_names]
    axes[1].bar(x + i*w, vals, w, label=name,
                color=colors_roc[i], alpha=0.85)

axes[1].set_xticks(x + w)
axes[1].set_xticklabels(['Accuracy','Precision','Recall','F1','ROC-AUC'])
axes[1].set_title('Metrics Comparison')
axes[1].legend()
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for best model (Random Forest)
best = results['Random Forest']
cm = confusion_matrix(y_test, best['y_pred'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Creditworthy','Creditworthy'],
            yticklabels=['Not Creditworthy','Creditworthy'])
plt.title('Random Forest — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print(classification_report(y_test, best['y_pred'],
      target_names=['Not Creditworthy','Creditworthy']))

In [ ]:
# Feature importance
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)\
             .sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='bar', color='#1a3c6e', alpha=0.85, edgecolor='white')
plt.title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Save best model
import os
os.makedirs('model', exist_ok=True)
with open('model/credit_model.pkl','wb') as f:
    pickle.dump(rf, f)
with open('model/scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

print('✅ Best model (Random Forest) saved!')
print(f'\n📊 FINAL RESULTS SUMMARY')
print('='*55)
for name, res in results.items():
    print(f'  {name:<22} Acc:{res["accuracy"]:.3f} F1:{res["f1"]:.3f} AUC:{res["roc_auc"]:.3f}')
print('='*55)
print('  🏆 Best Model: Random Forest')
print(f'  Author: Rose Sharma | CodeAlpha ML Internship')